# House Price Prediction — Detailed

**Tabular Regression Project** — Comprehensive house price prediction with detailed feature engineering.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Ames Housing (local `dataset/train.csv`)
Target: `SalePrice`

## 2 · Project Overview

The **Ames Housing** dataset is the modern successor to the Boston Housing dataset. With 79 features describing every aspect of residential homes in Ames, Iowa, it is the gold standard for learning regression feature engineering.

This detailed version focuses on comprehensive preprocessing and feature engineering across the many feature types.

## 3 · Learning Objectives

1. Handle a high-dimensional dataset (79 features).
2. Deal with mixed types: ordinal, nominal, continuous, discrete.
3. Handle significant missing data patterns.
4. Apply feature engineering at scale.
5. Compare models on a challenging real-world dataset.

## 4 · Problem Statement

Predict the **SalePrice** of residential properties in Ames, Iowa from 79 explanatory variables covering almost every aspect of the property.

## 5 · Why This Project Matters

- The Ames dataset is the Kaggle competition standard for regression.
- 79 features teach practical feature selection and engineering at scale.
- Real property assessor data — directly applicable to real estate valuation.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Local: `dataset/train.csv` (Ames Housing) |
| **Rows** | ~1,460 |
| **Features** | 79 (numeric + categorical) |
| **Target** | SalePrice (USD) |

## 7 · Dataset Source and License Notes

- **Source**: Ames Housing dataset by Dean De Cock, published in Journal of Statistics Education.
- **License**: Educational use.
- **Note**: Real property sales from Ames, Iowa. More detailed than Boston Housing.

## 8 · Environment Setup

In [1]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
print('All packages ready.')

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

Imports complete.


## 10 · Configuration / Constants

In [3]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'SalePrice'
np.random.seed(SEED)

## 11 · Dataset Loading

In [4]:
data_file = os.path.join(SAVE_DIR, 'dataset', 'train.csv')
assert os.path.exists(data_file), f'Not found: {data_file}'
df = pd.read_csv(data_file)
print(f'Loaded: {df.shape}')
print(f'Numeric: {len(df.select_dtypes(include="number").columns)}, Categorical: {len(df.select_dtypes(include="object").columns)}')
df.head()

Loaded: (1460, 81)
Numeric: 38, Categorical: 43


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## 12 · Data Validation Checks

In [5]:
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print('Columns with > 10% missing:')
print(missing_pct[missing_pct > 10])
print(f'\nTotal missing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')

Columns with > 10% missing:
PoolQC         99.520548
MiscFeature    96.301370
Alley          93.767123
Fence          80.753425
MasVnrType     59.726027
FireplaceQu    47.260274
LotFrontage    17.739726
dtype: float64

Total missing values: 7829
Duplicates: 0


## 13 · Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
df[TARGET].hist(bins=50, ax=axes[0,0], edgecolor='black')
axes[0,0].set_title('SalePrice Distribution')
if 'GrLivArea' in df.columns:
    axes[0,1].scatter(df['GrLivArea'], df[TARGET], alpha=0.3)
    axes[0,1].set_xlabel('GrLivArea'); axes[0,1].set_ylabel(TARGET)
    axes[0,1].set_title('Living Area vs Price')
if 'OverallQual' in df.columns:
    df.groupby('OverallQual')[TARGET].mean().plot.bar(ax=axes[1,0])
    axes[1,0].set_title('Avg Price by Overall Quality')
if 'YearBuilt' in df.columns:
    axes[1,1].scatter(df['YearBuilt'], df[TARGET], alpha=0.2)
    axes[1,1].set_xlabel('Year Built'); axes[1,1].set_ylabel(TARGET)
    axes[1,1].set_title('Year Built vs Price')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [7]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64

Skewness: 1.88


## 15 · Train / Validation / Test Split

## 16 · Preprocessing

Drop columns with >40% missing, impute the rest, encode categoricals.

In [8]:
from sklearn.preprocessing import LabelEncoder

# Drop columns with too many missing values
high_missing = [col for col in df.columns if df[col].isnull().sum() / len(df) > 0.40]
df = df.drop(columns=high_missing)
print(f'Dropped {len(high_missing)} high-missing columns: {high_missing}')

# Drop Id
if 'Id' in df.columns: df = df.drop(columns=['Id'])

# Encode categoricals
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna('None')
    df[col] = LabelEncoder().fit_transform(df[col])

# Fill numeric
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

print(f'Preprocessed: {df.shape}')

Dropped 6 high-missing columns: ['Alley', 'MasVnrType', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature']
Preprocessed: (1460, 74)


## 17 · Feature Engineering

In [9]:
# Add useful features
if 'YearBuilt' in df.columns:
    df['house_age'] = df['YrSold'].max() - df['YearBuilt'] if 'YrSold' in df.columns else 2010 - df['YearBuilt']
if 'TotalBsmtSF' in df.columns and 'GrLivArea' in df.columns:
    df['total_sf'] = df['TotalBsmtSF'] + df['GrLivArea']

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (1168, 75), Test: (292, 75)


## 18 · Baseline Model

In [10]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:,.0f}, MAE={baseline_mae:,.0f}, R²={baseline_r2:.4f}')

Baseline LR: RMSE=34,223, MAE=21,468, R²=0.8473


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

                               Adjusted R-Squared  R-Squared          RMSE  \
Model                                                                        
XGBRegressor                             0.873217   0.905893  26866.930156   
CatBoostRegressor                        0.867813   0.901882  27433.496278   
GradientBoostingRegressor                0.865672   0.900293  27654.807756   
ExtraTreesRegressor                      0.864629   0.899519  27761.912165   
PoissonRegressor                         0.855346   0.892628  28698.032669   
GammaRegressor                           0.851543   0.889805  29072.812654   
HistGradientBoostingRegressor            0.851369   0.889676  29089.904490   
LGBMRegressor                            0.849109   0.887999  29310.179656   
RandomForestRegressor                    0.847003   0.886435  29514.034376   
BaggingRegressor                         0.827051   0.871625  31379.526991   
HuberRegressor                           0.816013   0.863432  32

## 20 · FLAML AutoML Run

In [12]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R²: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

FLAML failed: XGBModel.fit() got an unexpected keyword argument 'callbacks'


## 21 · Boosting Models

In [13]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

CatBoost: RMSE=28455.84, MAE=16407.41, R²=0.8944
LightGBM: RMSE=29133.19, MAE=16924.29, R²=0.8893


XGBoost: RMSE=26726.32, MAE=16375.60, R²=0.9069


## 22 · Top 3 Model Selection

In [14]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

All models ranked by RMSE:
  XGBoost               RMSE=26726.32  MAE=16375.60  R²=0.9069
  CatBoost              RMSE=28455.84  MAE=16407.41  R²=0.8944
  LightGBM              RMSE=29133.19  MAE=16924.29  R²=0.8893
  Baseline_LR           RMSE=34223.41  MAE=21468.10  R²=0.8473

Top 3: ['XGBoost', 'CatBoost', 'LightGBM']


## 23 · Final Eval of Top 3

In [15]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R²={m['R2']:.4f}")

Top 3 Final Results:
XGBoost: RMSE=26726.32, MAE=16375.60, R²=0.9069
CatBoost: RMSE=28455.84, MAE=16407.41, R²=0.8944
LightGBM: RMSE=29133.19, MAE=16924.29, R²=0.8893


## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models.get('CatBoost', baseline)
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation and Business Insight

- **OverallQual** and **GrLivArea** are the top 2 predictors consistently.
- **total_sf** (engineered feature) captures value well.
- **Neighborhood** effects are significant even when encoded simply.
- Feature engineering matters more than model choice on this dataset.

## 26 · Limitations

- Ames, Iowa only — different markets have different drivers.
- ~1,460 rows limits complex model performance.
- Many features are ordinal but encoded as arbitrary integers.
- No time-series market trend modeling.

## 27 · How to Improve

1. Log-transform SalePrice.
2. Proper ordinal encoding for quality features.
3. Feature selection to reduce dimensionality.
4. Stacking ensemble of top models.

## 28 · Production Considerations

- Property valuation models need regular recalibration.
- Must comply with fair housing regulations.
- Explain predictions for appraisal transparency.
- Validate against comparable sales.

## 29 · Common Mistakes

1. Not handling the many missing-value patterns.
2. Dropping too many features prematurely.
3. Not engineering total square footage.
4. Ignoring the target's right skew.

## 30 · Mini Challenge

1. Predict log(SalePrice) and back-transform.
2. Use SHAP for feature importance.
3. Create quality × size interaction features.
4. Try PCA on the 79 features.

## 31 · Final Summary

- The Ames dataset is the best regression learning dataset available.
- Quality, size, and location features dominate predictions.
- Proper missing value handling and feature engineering are critical.
- Boosting models handle the high dimensionality well.

In [17]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')

Metrics saved to metrics.json

Notebook complete.
